# 10. Semantic Segmentation - BiSeNetV1

| 항목 | 내용 |
|---|---|
| 대상 모델 | `BiSeNetV1.dxnn` |
| 카테고리 | **Semantic Segmentation** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 부류 점수 대신 이미 판정된 부류 번호가 나오는 출력의 이점을 바이트 수로 계산해 제시할 수 있다.
2. 부류 번호 지도에서 특정 부류가 차지하는 화면 비율을 직접 계산할 수 있다.
3. 19개 부류 번호와 이름·색의 대응을 근거와 함께 설명할 수 있다.
4. 시각화가 0번 부류를 칠하지 않는 이유와 그 결과를 그림에서 지목할 수 있다.

## 1. Semantic Segmentation (의미 분할)

**영상의 픽셀 하나하나에 어느 부류에 속하는지를 붙이는 작업**

분류가 사진 한 장에 이름표 하나를 붙이는 것이라면, 이 작업은 **픽셀마다 이름표를 붙인다.** 도로와 인도의
경계처럼 형태가 정해지지 않은 영역도 화소 단위로 갈라낼 수 있다.

**출력의 성격**

픽셀마다 부류 번호가 하나씩 놓인 지도다. 같은 부류의 두 대상을 구분하지 않으므로 **차 세 대는 모두 하나의
`car` 영역**이 되고 개수를 셀 수 없다. 개체를 구분하려면 개체 분할이 필요하다.

**활용 사례**

- 자율주행의 주행 가능 영역 판정
- 위성·항공 영상의 토지 이용 구분
- 배경 분리·합성

**포함되지 않는 작업**

- 같은 부류의 개체 구분·개수 세기 (Instance Segmentation)
- 대상의 위치를 사각형으로 잡기 (Object Detection)
- 사진 한 장에 이름표 하나 (Image Classification)

## 2. 전체 파이프라인

```text
sample_parking.jpg  676×338 BGR
  → 전처리         BGR→RGB · 2048×1024 로 확대 · uint8 유지
  → run()          입력 Tensor 하나를 NPU 에 전달
  → Public Output  preds  [1, 1024, 2048]  uint16  (부류 번호가 이미 정해진 지도)
  → 후처리         이미 판정된 값이므로 축만 눌러 [1024, 2048] int32 로
  → Result         SegmentationResult (mask · width · height · class_ids)
  → 시각화         부류별 색 마스크를 원본 크기로 늘려 60% 로 덧씌움
```

![10. Semantic Segmentation 전체 파이프라인](resources/10_Semantic_Segmentation_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input_image` |
| Shape / Dtype | `[1, 1024, 2048, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | 직접 변경 (비율 유지 없음) |
| Normalization | 없음 |

**입력이 이 저장소에서 가장 크다**

- **6 MB 짜리 입력** — 2048×1024×3 이며 01번(147 KB)의 40배다. 픽셀마다 답을 내야 하므로 해상도를 줄이면
  경계가 뭉개진다
- **sample 은 오히려 작아서 확대된다** — 676×338 이 2048×1024 로 3.03배 커진다. 없는 화소를 보간해 만들어
  넣는 것이므로 경계의 정확도가 원본 해상도 이상으로 올라가지는 않는다
- **가로세로 비율이 2:1** — Cityscapes 주행 영상의 화면비를 따랐다. 세로로 긴 사진을 넣으면 크게 눌린다

### 3.2 출력

| Tensor | Shape / dtype | 의미 | Public |
|---|---|---|---|
| `preds` | `[1, 1024, 2048]` / `uint16` | 픽셀마다의 부류 번호 | O |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. 이 모델은 Task 가 `npu_0` 하나뿐이라
NPU 가 계산한 값이 그대로 나온다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 1024 | 높이 |
| 2 | 2048 | 너비 |

**부류 축이 없다.** 이것이 이 모델 출력의 핵심이다. 보통의 분할 모델은 부류마다 점수 한 장씩, 즉
`[1, 19, 1024, 2048]` `float32` 를 내놓고 그중 가장 큰 부류를 고르는 계산을 밖에서 한다. 이 배포본은 **그
고르는 계산까지 모델 안에서 끝내** 번호 한 장만 내보낸다.

| | 점수를 내보낼 때 | 번호만 내보낼 때 (이 모델) |
|---|---:|---:|
| shape | `[1, 19, 1024, 2048]` | `[1, 1024, 2048]` |
| dtype | `float32` | `uint16` |
| 바이트 | 159,383,552 | 4,194,304 |

**38배 차이다.** 값을 옮기는 시간과 메모리가 그만큼 줄어들고, 밖에서 할 계산도 없어진다. 대신 점수를 볼 수
없으므로 **어느 부류가 얼마나 근소하게 이겼는지 알 수 없다** — 경계에서 판정이 흔들리는 정도를 살펴보려면
점수를 내보내는 배포본이 필요하다.

`uint16` 을 쓰는 것은 번호가 0~18 이라 정수면 충분하기 때문이다. `uint8` 로도 담기지만 부류가 256개를
넘는 데이터셋까지 같은 형식으로 다루려면 16비트가 안전하다.

**19개 부류와 색**

번호의 정본은 Cityscapes 가 정한 `trainId` 이며, 이 데이터셋은 드물게 **번호까지 공식으로 정해 둔** 경우다.
색도 데이터셋이 정한 값을 그대로 쓴다.

| index | 이름 | RGB | index | 이름 | RGB |
|---:|---|---|---:|---|---|
| 0 | road | (128, 64, 128) | 10 | sky | (70, 130, 180) |
| 1 | sidewalk | (244, 35, 232) | 11 | person | (220, 20, 60) |
| 2 | building | (70, 70, 70) | 12 | rider | (255, 0, 0) |
| 3 | wall | (102, 102, 156) | 13 | car | (0, 0, 142) |
| 4 | fence | (190, 153, 153) | 14 | truck | (0, 0, 70) |
| 5 | pole | (153, 153, 153) | 15 | bus | (0, 60, 100) |
| 6 | traffic light | (250, 170, 30) | 16 | train | (0, 80, 100) |
| 7 | traffic sign | (220, 220, 0) | 17 | motorcycle | (0, 0, 230) |
| 8 | vegetation | (107, 142, 35) | 18 | bicycle | (119, 11, 32) |
| 9 | terrain | (152, 251, 152) | | | |

세로로 읽는 표가 아니다. 0~18 을 10개씩 두 묶음으로 옆에 늘어놓았다.

전체 목록: [Cityscapes label definitions](https://www.cityscapes-dataset.com/dataset-overview/)

**`person` 과 `rider` 가 나뉘어 있는 것**처럼 부류 구분이 데이터셋의 판단을 따른다. 자전거를 탄 사람은
`rider` 이고 걷는 사람은 `person` 이다. 이 구분을 모르면 결과를 잘못 읽는다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. 출력에 부류 축이 없고
dtype 이 `uint16` 인지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "BiSeNetV1.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 Cityscapes 19개 부류 이름을 읽는다. 이후 셀은 여기서 만든
이름을 그대로 쓴다.

In [ ]:
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

IMAGE_PATH = DX_APP / "sample" / "img/sample_parking.jpg"

# dx-all-suite 는 읽기 전용 원본이다. 바이트코드 캐시를 남기지 않도록 먼저 꺼 둔다.
sys.dont_write_bytecode = True
sys.path.insert(0, str(DX_APP / "src" / "python_example"))
from common.utility.labels import get_cityscapes_labels  # noqa: E402

CITYSCAPES_LABELS = get_cityscapes_labels()

print("classes", len(CITYSCAPES_LABELS), "/ index 13 =", CITYSCAPES_LABELS[13])

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값
- `SegmentationResult` — 후처리가 만들어 시각화로 넘기는 결과. 부류 번호 지도와 그 크기, 등장한 번호 목록을
  담는다
- `CITYSCAPES_PALETTE` — 부류별 색. 데이터셋이 정한 값이며 시각화에서만 쓴다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
CITYSCAPES_PALETTE = np.array([
    [128, 64, 128], [244, 35, 232], [70, 70, 70], [102, 102, 156], [190, 153, 153],
    [153, 153, 153], [250, 170, 30], [220, 220, 0], [107, 142, 35], [152, 251, 152],
    [70, 130, 180], [220, 20, 60], [255, 0, 0], [0, 0, 142], [0, 0, 70],
    [0, 60, 100], [0, 80, 100], [0, 0, 230], [119, 11, 32],
], dtype=np.uint8)


@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class SegmentationResult:
    """부류 번호 지도와 등장한 번호 목록. 원본 대응: common/base/i_processor.py"""

    mask: np.ndarray = field(default=None)
    width: int = 0
    height: int = 0
    class_ids: list = field(default_factory=list)
    class_names: list = field(default_factory=list)


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 출력 dtype 이 `uint16` 인 것을 확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

BGR 원본을 RGB 2048×1024 로 바꾼다. sample 이 모델 입력보다 작아 **확대**가 일어나며, 보간으로 만들어 넣은
화소는 원본에 없던 정보이므로 경계의 정확도를 높여 주지 않는다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 2048×1024 (원본보다 큼)
- dtype — `uint8` 유지
- 배율 — 가로·세로 모두 3.03배 확대

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}", figsize=(10, 5))

#### 전처리 함수 정의

- `bisenet_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `bisenet_preprocessor_process` — 이미지 한 장을 입력 Tensor 와 Context 로 변환

In [ ]:
def bisenet_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def bisenet_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (ctx.input_width, ctx.input_height),
                         interpolation=cv2.INTER_LINEAR)
    return resized, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. `scale_x` 와 `scale_y` 가 1 보다 큰 것이 확대라는 뜻이다.

In [ ]:
preprocessor_state = bisenet_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = bisenet_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"scale_x={ctx.scale_x:.4f} scale_y={ctx.scale_y:.4f}")
print(f"바이트        원본 {image.nbytes:,} → 입력 {input_tensor.nbytes:,}")

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 NPU 에 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복
비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, "min", array.min(), "max", array.max())

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- dtype — `uint16` 정수이므로 점수가 아니라 이미 정해진 번호다
- 값 범위 — 0~18 안에 들어오는지
- 등장한 번호 — 사진에 없는 부류는 나오지 않는지
- 바이트 — 점수를 내보낼 때와 비교

In [ ]:
raw = np.asarray(outputs[0])
print(f"raw           shape={raw.shape} dtype={raw.dtype} min={raw.min()} max={raw.max()}")
print(f"정수인가       {np.issubdtype(raw.dtype, np.integer)}")

unique_ids = np.unique(raw)
print(f"등장한 번호    {len(unique_ids)}개 / 19개 중")
print(f"바이트         이 출력 {raw.nbytes:,} / 점수로 내보냈다면 "
      f"{19 * INPUT_HEIGHT * INPUT_WIDTH * 4:,} ({19 * 4 / 2:.0f}배)")

### 4.4 후처리

**부류를 고르는 계산이 이미 끝났으므로 후처리가 할 일이 거의 없다.** 크기 1인 축을 눌러 `[1024, 2048]` 로
만들고 정수형으로 맞춘 뒤, 등장한 번호 목록을 세어 결과 객체에 담는다.

같은 후처리 코드가 점수를 내보내는 배포본도 받는다. 그래서 **출력의 차원 수와 dtype 을 보고 갈래를 정한다** —
정수형이거나 첫 축이 1이면 이미 판정된 값으로 보고 넘기며, 실수형 다차원이면 부류 축에서 가장 큰 자리를 고른다.

이 모델은 원본 크기로 되돌리는 단계를 켜지 않으므로 **부류 번호 지도가 모델 입력 크기로 남는다.** 원본 크기에
맞추는 일은 시각화가 색 마스크를 늘릴 때 한다.

#### 후처리 함수 정의

- `bisenet_postprocessor_create` — 부류 수를 state 에 담음
- `bisenet_postprocessor_process` — 축 정리·정수 변환·등장 번호 수집을 거쳐 Result 생성

In [ ]:
def bisenet_postprocessor_create(num_classes: int = 19) -> dict:
    """고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"num_classes": num_classes}


def bisenet_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    output = np.asarray(outputs[0])

    if output.ndim == 3 and (output.shape[0] == 1 or np.issubdtype(output.dtype, np.integer)):
        # 이미 부류가 정해진 값. 축만 눌러 쓴다.
        class_map = output.squeeze().astype(np.int32)
    else:
        # 점수를 내보내는 배포본이면 부류 축에서 가장 큰 자리를 고른다.
        class_map = np.argmax(output if output.ndim == 3 else output[0], axis=0).astype(np.int32)

    height, width = class_map.shape
    return [SegmentationResult(mask=class_map, width=width, height=height,
                               class_ids=np.unique(class_map).tolist())]

#### 후처리 실행

확인할 것:

- `mask` — `int32` `[1024, 2048]`
- `class_ids` — 등장한 번호와 그 이름
- 부류별 화면 비율 — 어느 부류가 사진을 얼마나 차지하는지

In [ ]:
postprocessor_state = bisenet_postprocessor_create()
results = bisenet_postprocessor_process(postprocessor_state, outputs, ctx)
result = results[0]

print(f"mask          shape={result.mask.shape} dtype={result.mask.dtype} "
      f"{result.width}x{result.height}")
print(f"class_ids     {result.class_ids}")

pixel_total = result.mask.size
ids, counts = np.unique(result.mask, return_counts=True)
print("\n부류별 화면 비율 (많은 순):")
for class_id, count in sorted(zip(ids.tolist(), counts.tolist()), key=lambda pair: -pair[1]):
    name = CITYSCAPES_LABELS[class_id] if class_id < len(CITYSCAPES_LABELS) else f"class_{class_id}"
    print(f"  {class_id:2d} {name:14s} {count:9,}  {count / pixel_total * 100:5.2f}%")

### 4.5 시각화

#### Visualizer 코드

부류 번호마다 정해진 색을 칠해 마스크를 만들고 원본 크기로 늘려 60% 투명도로 덧씌운다. **0번은 배경으로 보아
칠하지 않으므로 `road` 영역은 원본 색으로 남는다** — 도로가 칠해지지 않은 것은 판정이 없었다는 뜻이 아니다.

In [ ]:
def bisenet_visualizer_visualize(image: np.ndarray, results: list, alpha: float = 0.6,
                                 background_class: int = 0) -> np.ndarray:
    """부류별 색 마스크를 원본 위에 덧씌운다. 원본 대응: Runner.visualize()"""
    output = image.copy()
    if not results or results[0].mask is None or results[0].mask.size == 0:
        return output

    class_map = results[0].mask
    height, width = output.shape[:2]
    colored_small = np.zeros((class_map.shape[0], class_map.shape[1], 3), dtype=np.uint8)

    for class_id in range(len(CITYSCAPES_PALETTE)):
        if class_id == background_class:
            continue
        selected = class_map == class_id
        if np.any(selected):
            colored_small[selected] = CITYSCAPES_PALETTE[class_id]

    colored = cv2.resize(colored_small, (width, height), interpolation=cv2.INTER_LINEAR)
    cv2.addWeighted(output, 1 - alpha, colored, alpha, 0, dst=output)
    return output

#### 결과 시각화

색 마스크를 덧씌운 결과를 inline 으로 표시한다. 팔레트가 RGB 순서이고 그림은 BGR 이라 색이 데이터셋 표기와
뒤바뀌어 보이는데, 원본 코드가 변환 없이 그대로 쓰므로 같은 결과를 남긴다.

In [ ]:
output_image = bisenet_visualizer_visualize(image, results)
notebook_show_bgr(output_image, "Segmentation overlay (class 0 not painted)", figsize=(11, 6))

#### 부류 하나만 골라 보기

부류 번호 지도가 있으면 원하는 부류만 남길 수 있다. `car`(13) 에 해당하는 화소만 흰색으로 칠해 보면 같은
부류의 차 여러 대가 **하나의 덩어리로 이어져 개수를 셀 수 없다는 사실**이 드러난다.

In [ ]:
CAR_ID = 13
car_mask = (result.mask == CAR_ID).astype(np.uint8) * 255
car_mask_resized = cv2.resize(car_mask, (image.shape[1], image.shape[0]),
                              interpolation=cv2.INTER_NEAREST)

print(f"car 화소 {int((result.mask == CAR_ID).sum()):,}개 "
      f"({(result.mask == CAR_ID).mean() * 100:.2f}%)")
notebook_show_bgr(cv2.hconcat([cv2.cvtColor(car_mask_resized, cv2.COLOR_GRAY2BGR), image]),
                  "car class only (left) vs input (right)", figsize=(13, 4))